# Resume Optimizer - QLoRA Fine-tuned


# Dependencies and libraries 

In [ ]:
# Install Google Chrome manually via .deb file
# !wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
# !apt-get install -y ./google-chrome-stable_current_amd64.deb


In [3]:
# !pip install selenium webdriver-manager beautifulsoup4

  Using cached selenium-4.38.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached trio-0.32.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached soupsieve-2.8-py3-none-any.whl.metadata (4.6 kB)
Using cached selenium-4.38.0-py3-none-any.whl (9.7 MB)
Using cached trio-0.32.0-py3-none-any.whl (512 kB)
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached websocket_client-1.9.0-py3-none-any.whl (82 kB)
Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)
Using cach

In [1]:
# !pip install pandas numpy tqdm requests PyPDF2 python-docx selenium webdriver-manager beautifulsoup4 lxml pywin32 pyarrow google-genai

In [1]:
import os
import json
import re
import pandas as pd
import numpy as np

import time
from datetime import datetime
from tqdm import tqdm
import logging
import requests

from pathlib import Path
from typing import Optional, Dict, Any, Tuple


from PyPDF2 import PdfReader
from PyPDF2.errors import PdfReadError
from docx import Document
from docx.opc.constants import RELATIONSHIP_TYPE as RT
from docx.opc.exceptions import PackageNotFoundError

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

import base64
from google import genai
from google.genai import types



# Dataset Creation from 1800+ Resumes

This section describes the process of creating a dataset from over 1000 resumes. The dataset will be used for analysis, such as extracting key information like skills, experience, and education.

In [ ]:
try:
    import win32com.client as win32  # type: ignore
    import pywintypes  # type: ignore
except ImportError:
    win32 = None
    pywintypes = None

RECOVERABLE_EXTRACTION_ERRORS = (
    ValueError,
    OSError,
    PdfReadError,
    PackageNotFoundError,
    KeyError,
    RuntimeError,
)
# if pywintypes is not None and hasattr(pywintypes, "com_error"):
#     RECOVERABLE_EXTRACTION_ERRORS = RECOVERABLE_EXTRACTION_ERRORS + (
#         pywintypes.com_error,  # type: ignore[attr-defined]
#     )    

In [ ]:
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".doc"}
DEFAULT_INPUT_PATH = Path(r"C:\Users\Abhinav\Documents\Repo\Colab-proj-2\data\resumes")
DEFAULT_OUTPUT_PATH = Path("dataset/resume_text.jsonl")

In [ ]:
class WordAutomationClient:
    """Thin wrapper around Word COM automation to read legacy .doc files."""

    def __init__(self) -> None:
        if win32 is None:
            raise ImportError("pywin32 is required for .doc support on Windows")
        self._word = win32.Dispatch("Word.Application")
        self._word.Visible = False

    def close(self) -> None:
        if self._word is not None:
            self._word.Quit()
            self._word = None

    def __enter__(self) -> "WordAutomationClient":
        return self

    def __exit__(self, exc_type, exc, exc_tb) -> None:  # type: ignore[override]
        self.close()

    def extract_doc(self, file_path: Path) -> tuple[str, bool]:
        if self._word is None:
            raise RuntimeError("Word automation client is closed")
        document = self._word.Documents.Open(str(file_path))
        try:
            text = document.Content.Text
            contains_images = bool(document.InlineShapes.Count or document.Shapes.Count)
        finally:
            document.Close(False)
        return text.strip(), contains_images


def extract_text_from_pdf(file_path: Path) -> str:
    """Extract text from a PDF file."""
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"
    return text.strip()


def extract_text_from_docx(file_path: Path) -> str:
    """Extract text from a DOCX file."""
    doc = Document(file_path)
    text = ""
    for para in doc.paragraphs:
        text += para.text + "\n"
    return text.strip()


def extract_text(file_path: Path) -> str:
    """Extract text from a document file based on its extension."""
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        return extract_text_from_pdf(file_path)
    if suffix == ".docx":
        return extract_text_from_docx(file_path)
    raise ValueError(f"Unsupported file type: {file_path.suffix}")


def _xobject_dict_contains_images(x_objects) -> bool:
    if not x_objects:
        return False
    # Dereference x_objects if it's an indirect object
    try:
        x_objects = x_objects.get_object()
    except AttributeError:
        pass
    if not hasattr(x_objects, "values"):
        return False
    for obj in x_objects.values():
        try:
            x_obj = obj.get_object()
        except AttributeError:
            x_obj = obj
        subtype = x_obj.get("/Subtype")
        if subtype == "/Image":
            return True
        if subtype == "/Form":
            child_resources = x_obj.get("/Resources")
            child_x_objects = None
            if child_resources:
                # Dereference indirect objects
                try:
                    child_resources = child_resources.get_object()
                except AttributeError:
                    pass
                if hasattr(child_resources, "get"):
                    child_x_objects = child_resources.get("/XObject")
            if _xobject_dict_contains_images(child_x_objects):
                return True
    return False


def pdf_has_images(file_path: Path) -> bool:
    reader = PdfReader(file_path)
    for page in reader.pages:
        if hasattr(page, "images") and page.images:
            return True
        resources = page.get("/Resources")
        if not resources:
            continue
        # Dereference indirect objects
        try:
            resources = resources.get_object()
        except AttributeError:
            pass
        if not hasattr(resources, "get"):
            continue
        x_objects = resources.get("/XObject")
        if _xobject_dict_contains_images(x_objects):
            return True
    return False


def docx_has_images(file_path: Path) -> bool:
    doc = Document(file_path)
    return any(rel.reltype == RT.IMAGE for rel in doc.part.rels.values())


def has_images(file_path: Path) -> bool:
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        return pdf_has_images(file_path)
    if suffix == ".docx":
        return docx_has_images(file_path)
    raise ValueError(f"Unsupported file type: {file_path.suffix}")


def collect_documents(target: Path) -> List[Path]:
    if target.is_file():
        return [target]
    if target.is_dir():
        return sorted(
            [f for f in target.rglob("*") if f.suffix.lower() in SUPPORTED_EXTENSIONS]
        )
    raise FileNotFoundError(f"'{target}' is not a valid file or directory.")


def process_document(
    file_path: Path, word_client: Optional["WordAutomationClient"]
) -> Dict[str, str | bool]:
    suffix = file_path.suffix.lower()
    if suffix == ".doc":
        if word_client is None:
            raise RuntimeError(
                ".doc support requires Microsoft Word and pywin32; both appear unavailable."
            )
        resume_text, contains_images = word_client.extract_doc(file_path)
    else:
        resume_text = extract_text(file_path)
        contains_images = has_images(file_path)

    return {
        "filename": file_path.name,
        "filetype": suffix,
        "resume_text": resume_text,
        "has_images": contains_images,
    }


def main() -> None:
    # Update these paths to control which resumes are processed and where JSONL output lands.
    input_path = DEFAULT_INPUT_PATH
    output_path = DEFAULT_OUTPUT_PATH

    try:
        documents = collect_documents(input_path)
    except FileNotFoundError as exc:
        print(exc)
        return

    if not documents:
        print(f"No PDF/DOC/DOCX files found under '{input_path}'.")
        return

    needs_word = any(path.suffix.lower() == ".doc" for path in documents)
    word_client: Optional[WordAutomationClient] = None
    if needs_word:
        try:
            word_client = WordAutomationClient()
        except ImportError as exc:
            print(f"Cannot open .doc files: {exc}")
            return

    processed_count = 0
    output_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with output_path.open("w", encoding="utf-8") as fh:
            for doc_path in documents:
                try:
                    record = process_document(doc_path, word_client)
                except RECOVERABLE_EXTRACTION_ERRORS as exc:
                    print(f"Skipping {doc_path}: {exc}")
                    continue
                fh.write(json.dumps(record, ensure_ascii=False) + "\n")
                processed_count += 1
                print(f"Processed {doc_path.name}")
    finally:
        if word_client is not None:
            word_client.close()

    if processed_count:
        print(
            f"Wrote metadata for {processed_count} file(s) to '{output_path.as_posix()}'"
        )
    else:
        print("No files were successfully processed; see logs above for details.")



In [ ]:
# running the main function
main()

# Building a Job Scraper

This section outlines the development of a job scraper to collect job postings from various sources. The scraper will extract relevant information such as job titles, descriptions, requirements, and company details to complement the resume dataset for analysis or matching purposes.

In [ ]:
# import requests

try:
    response = requests.get('https://api64.ipify.org?format=json')
    response.raise_for_status() # Raise an exception for HTTP errors
    ip_data = response.json()
    ip_address = ip_data.get('ip')
    if ip_address:
        print(f"Your IP Address is: {ip_address}")
    else:
        print("Could not retrieve IP address from the service.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching IP address: {e}")
except ValueError:
    print("Error decoding JSON response.")

In [ ]:
# from selenium import webdriver
# from selenium.webdriver.chrome.service import Service
# from webdriver_manager.chrome import ChromeDriverManager
# from bs4 import BeautifulSoup

# import pandas as pd
# import time
# import logging
# import os
# from datetime import datetime
# from tqdm import tqdm

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
class JobScraper:
    """Handles web scraping of job descriptions from various job posting websites."""

    def __init__(self, headless=True, wait_time=5, delay=2):
        """
        Initialize the job description scraper.

        Args:
            headless (bool): Run browser in headless mode
            wait_time (int): Time to wait for page to load (seconds)
            delay (int): Delay between requests (seconds)
        """
        self.headless = headless
        self.wait_time = wait_time
        self.delay = delay
        self.driver = None

    def setup_driver(self):
        """Set up Chrome WebDriver with appropriate options."""
        options = webdriver.ChromeOptions()
        if self.headless:
            options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')

        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=options
        )
        return self.driver

    def close_driver(self):
        """Close the WebDriver if it exists."""
        if self.driver:
            self.driver.quit()
            self.driver = None

    def scrape_job(self, job_url):
        """
        Extract job description from a job posting URL.

        Args:
            job_url (str): URL of the job posting

        Returns:
            tuple: (job_description, status)
        """
        if not self.driver:
            self.setup_driver()

        try:
            self.driver.get(job_url)
            time.sleep(self.wait_time)

            soup = BeautifulSoup(self.driver.page_source, 'html.parser')

            body_text = soup.body.get_text() if soup.body else ""

            # Extract the job description part
            # Assuming it starts after job details like Full-time, Onsite, etc.
            lines = body_text.split('\n')
            desc_start = False
            description = []

            for line in lines:
                line = line.strip()
                if any(keyword in line for keyword in ['Full-time', 'Onsite', 'Remote', 'Hybrid', 'Part-time']):
                    desc_start = True
                if desc_start and line:
                    description.append(line)

            job_text = ' '.join(description)

            if len(job_text) < 100:
                return None, "Description too short (< 100 chars)"

            # Truncate very long descriptions
            if len(job_text) > 10000:
                job_text = job_text[:10000] + "..."

            return job_text, "Success"

        except Exception as e:
            logger.error(f"Error scraping URL {job_url}: {str(e)}")
            return None, f"Error: {str(e)}"

    def scrape_from_csv(self, input_csv, output_csv=None):
        """Scrape jobs from CSV file with URLs."""

        if not os.path.exists(input_csv):
            logger.error(f"Input CSV file not found: {input_csv}")
            return None

        # Read input CSV
        try:
            df = pd.read_csv(input_csv)
        except Exception as e:
            logger.error(f"Error reading CSV: {e}")
            return None

        # Find URL column - look for 'Apply', 'url', or 'link' columns
        url_column = None

        # Priority order: Apply > URL > Link
        column_priorities = ['apply', 'url', 'link']

        for priority_col in column_priorities:
            for col in df.columns:
                if priority_col in col.lower():
                    url_column = col
                    break
            if url_column:
                break

        if url_column is None:
            logger.error("No URL column found in CSV. Expected column with 'Apply', 'URL', or 'Link' in name.")
            logger.info(f"Available columns: {list(df.columns)}")
            return None

        logger.info(f"Found {len(df)} URLs to scrape in column '{url_column}'")

        # Create output filename if not provided
        if output_csv is None:
            timestamp = datetime.now().strftime("%H%M%S%m%d%Y")
            output_csv = f"data/scraped_jobs_{timestamp}.csv"
            os.makedirs("data", exist_ok=True)

        # Create CSV file with headers if it doesn't exist
        file_exists = os.path.exists(output_csv)
        if not file_exists:
            with open(output_csv, 'w', encoding='utf-8', newline='') as f:
                f.write('URL,Job Description,Scrape Status\n')

        # Scrape each URL
        successful = 0
        failed = 0

        try:
            with tqdm(total=len(df), desc="Scraping jobs") as pbar:
                for idx, row in df.iterrows():
                    job_url = row[url_column]

                    if pd.isna(job_url) or not job_url.strip():
                        # Save immediately to CSV
                        result_df = pd.DataFrame([[job_url, "", "Empty URL"]],
                                                columns=['URL', 'Job Description', 'Scrape Status'])
                        result_df.to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8')
                        failed += 1
                        pbar.update(1)
                        continue

                    # Scrape job
                    description, status = self.scrape_job(job_url)

                    # Save immediately to CSV after each scrape
                    result_df = pd.DataFrame([[
                        job_url,
                        description if description else "",
                        status
                    ]], columns=['URL', 'Job Description', 'Scrape Status'])

                    result_df.to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8')

                    if status == "Success":
                        successful += 1
                    else:
                        failed += 1

                    pbar.set_postfix({
                        'Success': successful,
                        'Failed': failed,
                        'Rate': f"{successful/(successful+failed)*100:.1f}%" if (successful+failed) > 0 else "0%"
                    })

                    # Delay between requests
                    time.sleep(self.delay)
                    pbar.update(1)

            logger.info(f"Results saved to {output_csv}")
            logger.info(f"Summary: {successful} successful, {failed} failed")
            return output_csv,result_df

        except Exception as e:
            logger.error(f"Error during scraping: {e}")
            logger.info(f"Partial results saved to {output_csv}")
            logger.info(f"Summary before error: {successful} successful, {failed} failed")
            return output_csv,result_df
        finally:
            self.close_driver()

# Cleaning and Combining the Scraped Job and Resume Data
This section focuses on processing and integrating the scraped job descriptions with the resume dataset. Key steps include:

- Loading the cleaned scraped job data from CSV and the resume text data from JSONL.
- Filtering and cleaning the resume data to remove entries with images and normalize text.
- Randomly assigning job descriptions to resumes to create a balanced dataset for potential matching or analysis tasks.
- Saving the combined dataset in multiple formats (CSV, Parquet, JSONL) for flexibility and efficiency.
- Comparing file sizes and data integrity across formats to ensure consistency.
- The resulting dataset pairs each resume with a job description, facilitating downstream tasks like resume-job matching or machine learning model training. Note that Parquet offers the most compact storage, while JSONL preserves structure for streaming.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

input_csv = "dataset/merged_csv.csv"
output_folder = 'dataset/scrape_job/'

In [ ]:


# Ensure the Google Drive output folder exists
os.makedirs(output_folder, exist_ok=True)

# Updated output_csv to save to Google Drive
timestamp = datetime.now().strftime("%H%M%S%m%d%Y")
output_csv = os.path.join(output_folder, f"scraped_jobs_{timestamp}.csv")

scraper = JobScraper()
print("Starting job scraping...")
result_file = scraper.scrape_from_csv(input_csv, output_csv)

if result_file:
    print(f"Scraping complete! Results saved to {result_file}")
else:
    print("Scraping failed!")


In [ ]:

df_jobs = pd.read_csv('dataset/cleaned_scraped_jobs_21390711212025.csv')
df_resumes = pd.read_json('dataset/resume_text.jsonl', lines=True)

In [ ]:
df_resumes = df_resumes[~((df_resumes['has_images'] == True))]


# Function to clean text by removing non-ASCII characters and extra whitespace
def clean_text(text):
    # Remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning to resume_text and filename columns
df_resumes['resume_text'] = df_resumes['resume_text'].apply(clean_text)
df_resumes['filename'] = df_resumes['filename'].apply(clean_text)

# Optionally, filter out rows where resume_text is empty or too short after cleaning
df_resumes = df_resumes[df_resumes['resume_text'].str.len() > 10]  # Example: keep rows with more than 10 characters

print(f"Resumes DataFrame shape after cleaning: {df_resumes.shape}")

# Clean the resume_text column by replacing any sequence of whitespace (including newlines, tabs, etc.) with a single space
df_resumes['resume_text'] = df_resumes['resume_text'].str.replace(r'[^\w\s]', '', regex=True).str.replace(r'\s+', ' ', regex=True)


In [13]:
df_resumes.head()
df_resumes.info()

df_jobs.head()
df_jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   URL              249 non-null    object
 1   Job Description  249 non-null    object
 2   Scrape Status    249 non-null    object
dtypes: object(3)
memory usage: 6.0+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   URL              249 non-null    object
 1   Job Description  249 non-null    object
 2   Scrape Status    249 non-null    object
dtypes: object(3)
memory usage: 6.0+ KB


In [ ]:


# Get all job descriptions
job_descriptions = df_jobs['Job Description'].values

# Calculate how many times each job description should be used (approximately)
n_resumes = len(df_resumes)
n_jobs = len(job_descriptions)

# Create an array of job indices that will be evenly distributed
# Repeat the job indices enough times to cover all resumes
repeats = (n_resumes // n_jobs) + 1
job_indices = np.tile(np.arange(n_jobs), repeats)[:n_resumes]

# Shuffle to randomize the assignment
np.random.seed(42)  # For reproducibility
np.random.shuffle(job_indices)

# Assign job descriptions to resumes
df_resumes['job_description'] = job_descriptions[job_indices]

# Verify the distribution
print(f"Total resumes: {len(df_resumes)}")
print(f"Unique job descriptions: {df_resumes['job_description'].nunique()}")
print(f"\nDistribution of job descriptions (counts):")
print(df_resumes['job_description'].value_counts().describe())
print(f"\nMin count: {df_resumes['job_description'].value_counts().min()}")
print(f"Max count: {df_resumes['job_description'].value_counts().max()}")
print(f"\nDataFrame info:")
df_resumes.info()

Total resumes: 1565
Unique job descriptions: 245

Distribution of job descriptions (counts):
count    245.000000
mean       6.387755
std        0.882812
min        6.000000
25%        6.000000
50%        6.000000
75%        7.000000
max       13.000000
Name: count, dtype: float64

Min count: 6
Max count: 13

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
Index: 1565 entries, 0 to 2126
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   filename         1565 non-null   object
 1   filetype         1565 non-null   object
 2   resume_text      1565 non-null   object
 3   has_images       1565 non-null   bool  
 4   job_description  1565 non-null   object
dtypes: bool(1), object(4)
memory usage: 62.7+ KB


In [ ]:
DATA_DIR = ".//dataset"
df_resumes.to_csv(os.path.join(DATA_DIR, 'resume_dataset.csv'), index=False)
df_resumes.to_parquet(os.path.join(DATA_DIR, 'resume_dataset.parquet'), engine='pyarrow', compression='snappy')
df_resumes.to_json(os.path.join(DATA_DIR, 'resume_dataset.jsonl'), orient='records', lines=True)

In [ ]:
# Read the saved files and compare them


DATA_DIR = ".//dataset"

# Read all three formats
df_csv = pd.read_csv(os.path.join(DATA_DIR, 'resume_dataset.csv'))
df_parquet = pd.read_parquet(os.path.join(DATA_DIR, 'resume_dataset.parquet'), engine='pyarrow')
df_jsonl = pd.read_json(os.path.join(DATA_DIR, 'resume_dataset.jsonl'), lines=True)

# Compare shapes
print("=== Shape Comparison ===")
print(f"CSV:     {df_csv.shape}")
print(f"Parquet: {df_parquet.shape}")
print(f"JSONL:   {df_jsonl.shape}")

# Compare columns
print("\n=== Columns Comparison ===")
print(f"CSV columns:     {list(df_csv.columns)}")
print(f"Parquet columns: {list(df_parquet.columns)}")
print(f"JSONL columns:   {list(df_jsonl.columns)}")

# Compare data types
print("\n=== Data Types Comparison ===")
print("CSV dtypes:")
print(df_csv.dtypes)
print("\nParquet dtypes:")
print(df_parquet.dtypes)
print("\nJSONL dtypes:")
print(df_jsonl.dtypes)

# Compare file sizes
print("\n=== File Size Comparison ===")
csv_size = os.path.getsize(os.path.join(DATA_DIR, 'resume_dataset.csv'))
parquet_size = os.path.getsize(os.path.join(DATA_DIR, 'resume_dataset.parquet'))
jsonl_size = os.path.getsize(os.path.join(DATA_DIR, 'resume_dataset.jsonl'))

print(f"CSV:     {csv_size / (1024*1024):.2f} MB")
print(f"Parquet: {parquet_size / (1024*1024):.2f} MB")
print(f"JSONL:   {jsonl_size / (1024*1024):.2f} MB")

# Check if data is identical
print("\n=== Data Equality Check ===")
print(f"CSV == Parquet: {df_csv.equals(df_parquet)}")
print(f"CSV == JSONL:   {df_csv.equals(df_jsonl)}")
print(f"Parquet == JSONL: {df_parquet.equals(df_jsonl)}")

# Show sample data
print("\n=== Sample Data (first 2 rows) ===")
df_csv.head(2)

=== Shape Comparison ===
CSV:     (1565, 5)
Parquet: (1565, 5)
JSONL:   (1565, 5)

=== Columns Comparison ===
CSV columns:     ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']
Parquet columns: ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']
JSONL columns:   ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']

=== Data Types Comparison ===
CSV dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

Parquet dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

JSONL dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

=== File Size Comparison ===
CSV:     25.36 MB
Parquet: 9.48 MB
JSONL:   25.50 MB

=== Data Equality Check ===
C

,filename,filetype,resume_text,has_images,job_description
0,00_Willie_Ellis_Go_Python.docx,.docx,Willie Ellis Senior Software Engineer Buffalo ...,False,Hanger/Textiles jobs in United StatesOverviewC...
1,10272022 Resume.docx,.docx,SUMMARY Leverage my skills education and exper...,False,Production Associate - Garment Hanger/Inspecto...


In [18]:
DATA_DIR = ".//dataset"
df = pd.read_parquet(os.path.join(DATA_DIR, 'resume_dataset.parquet'), engine='pyarrow')

# Ollama Resume Generator

This section uses a local Ollama model to generate tailored resumes based on the resume text and job descriptions.

In [19]:
# JSON Schema for tailored resume output
RESUME_SCHEMA = '''{"$schema":"http://json-schema.org/draft-04/schema#","type":"object","properties":{"personal_information":{"type":"object","properties":{"name":{"type":"string"},"email":{"type":"string"},"phone":{"type":"string"},"location":{"type":"string"},"socials":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]}},"required":["name","email","phone","location"]},"summary":{"type":"string"},"experiences":{"type":"array","items":[{"type":"object","properties":{"designation":{"type":"string"},"companyName":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"points":{"type":"array","items":[{"type":"string"}]}},"required":["designation","companyName","location","start_date"]}]},"education":{"type":"array","items":[{"type":"object","properties":{"institution":{"type":"string"},"degree":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"gpa":{"type":"string"}},"required":["institution","degree","location","start_date","gpa"]}]},"skills":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"data":{"type":"array","items":[{"type":"string"}]}},"required":["name","data"]}]},"projects":{"type":"array","items":[{"type":"object","properties":{"projectName":{"type":"string"},"caption":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"url":{"type":"string"},"projectDetails":{"type":"array","items":[{"type":"string"}]},"externalSources":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]},"technologiesUsed":{"type":"array","items":[{"type":"string"}]}},"required":["projectName","location","projectDetails"]}]},"certifications":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"issuing_organization":{"type":"string"},"issue_date":{"type":"string"},"expiration_date":{"type":"string"},"credential_id":{"type":"string"},"url":{"type":"string"}},"required":["name","issuing_organization","issue_date","expiration_date","credential_id","url"]}]},"awards":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"extracurricular_achievements":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"languages":{"type":"array","items":[{"type":"object","properties":{"language":{"type":"string"},"proficiency":{"type":"string"}},"required":["language","proficiency"]}]}},"required":["personal_information","education","skills","extracurricular_achievements"]}'''

# Load prompt template
with open('dataset/prompt.txt', 'r', encoding='utf-8') as f:
    PROMPT_TEMPLATE = f.read()

print("Prompt template loaded successfully")
print(f"Schema defined:\n{RESUME_SCHEMA[:200]}...")
print(f"Prompt template preview:\n{PROMPT_TEMPLATE[:500]}...")

Prompt template loaded successfully
Schema defined:
{"$schema":"http://json-schema.org/draft-04/schema#","type":"object","properties":{"personal_information":{"type":"object","properties":{"name":{"type":"string"},"email":{"type":"string"},"phone":{"ty...
Prompt template preview:
You create a tailored resume based on the job description.

Your task:
1. Read the RESUME_TEXT.
2. Read the JOB_DESCRIPTION.
3. Use only the information inside these two.
4. Follow the SCHEMA exactly.
5. Write a tailored resume in JSON using the SCHEMA.
6. Do not output anything outside the JSON.
7. If a field is missing in the resume, write a short, safe placeholder that fits the job.


RESUME_TEXT:
"""
<<PASTE RESUME HERE>>
"""

JOB_DESCRIPTION:
"""
<<PASTE JD HERE>>
"""

SCHEMA:
"""
<<PASTE J...


In [ ]:


class OllamaResumeGenerator:
    """Class to generate tailored resumes using local Ollama model (supports thinking models)."""
    
    def __init__(
        self,
        model_name: str = "qwen3:8b",
        base_url: str = "http://localhost:11434",
        output_path: str = "dataset/tailored_resumes",  # Base name without extension
        timeout: int = 300,  # Increased for thinking models
        max_retries: int = 3,
        enable_thinking: bool = True  # Enable thinking mode for supported models
    ):
        """
        Initialize the Ollama Resume Generator.
        
        Args:
            model_name: Name of the Ollama model to use
            base_url: Base URL for the Ollama API
            output_path: Base path for output file (timestamp will be added)
            timeout: Request timeout in seconds (higher for thinking models)
            max_retries: Maximum number of retries on failure
            enable_thinking: Whether to enable thinking mode (adds /think suffix)
        """
        self.model_name = model_name
        self.base_url = base_url
        self.api_url = f"{base_url}/api/generate"
        self.timeout = timeout
        self.max_retries = max_retries
        self.enable_thinking = enable_thinking
        
        # Add timestamp to output filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.output_path = Path(f"{output_path}_{timestamp}.jsonl")
        
        # Ensure output directory exists
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Store session start time
        self.session_start = datetime.now().isoformat()
        
    def _build_prompt(self, resume_text: str, job_description: str) -> str:
        """Build the prompt by filling in the template."""
        prompt = PROMPT_TEMPLATE.replace("<<PASTE RESUME HERE>>", resume_text)
        prompt = prompt.replace("<<PASTE JD HERE>>", job_description)
        prompt = prompt.replace("<<PASTE JSON SCHEMA HERE>>", RESUME_SCHEMA)
        return prompt
    
    def _extract_thinking_and_response(self, response: str) -> Tuple[Optional[str], str]:
        """
        Extract thinking content and actual response from thinking model output.
        
        Returns:
            Tuple of (thinking_content, actual_response)
        """
        if not response:
            return None, ""
        
        thinking_content = None
        actual_response = response
        
        # Pattern to match <think>...</think> blocks
        think_pattern = r'<think>(.*?)</think>'
        think_match = re.search(think_pattern, response, re.DOTALL)
        
        if think_match:
            thinking_content = think_match.group(1).strip()
            # Remove the thinking block from response
            actual_response = re.sub(think_pattern, '', response, flags=re.DOTALL).strip()
        
        return thinking_content, actual_response
    
    def _call_ollama(self, prompt: str) -> Tuple[Optional[str], Optional[str]]:
        """
        Make a request to the Ollama API.
        
        Returns:
            Tuple of (response, thinking_content)
        """
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False,
            # "options": {
            #     "temperature": 0.7,
            #     "num_predict": 4096  # Increased for thinking models
            # }
        }
        
        for attempt in range(self.max_retries):
            try:
                response = requests.post(
                    self.api_url,
                    json=payload,
                    timeout=self.timeout
                )
                response.raise_for_status()
                result = response.json()
                raw_response = result.get("response", "")
                
                # Extract thinking and actual response
                thinking, actual_response = self._extract_thinking_and_response(raw_response)
                
                return actual_response, thinking
                
            except requests.exceptions.Timeout:
                print(f"Timeout on attempt {attempt + 1}/{self.max_retries}")
            except requests.exceptions.RequestException as e:
                print(f"Request error on attempt {attempt + 1}/{self.max_retries}: {e}")
            
            if attempt < self.max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
        
        return None, None
    
    def _extract_json(self, response: str) -> Optional[Dict[str, Any]]:
        """Extract JSON from the model response."""
        if not response:
            return None
        
        # Try to find JSON in the response
        response = response.strip()
        
        # Try direct parsing first
        try:
            return json.loads(response)
        except json.JSONDecodeError:
            pass
        
        # Try to extract JSON from markdown code blocks
        if "```json" in response:
            start = response.find("```json") + 7
            end = response.find("```", start)
            if end > start:
                try:
                    return json.loads(response[start:end].strip())
                except json.JSONDecodeError:
                    pass
        
        # Try generic code blocks
        if "```" in response:
            start = response.find("```") + 3
            # Skip language identifier if present
            newline_pos = response.find("\n", start)
            if newline_pos > start:
                start = newline_pos + 1
            end = response.find("```", start)
            if end > start:
                try:
                    return json.loads(response[start:end].strip())
                except json.JSONDecodeError:
                    pass
        
        # Try to extract JSON between curly braces
        start = response.find("{")
        end = response.rfind("}") + 1
        if start >= 0 and end > start:
            try:
                return json.loads(response[start:end])
            except json.JSONDecodeError:
                pass
        
        return None
    
    def generate_single(self, resume_text: str, job_description: str, filename: str) -> Dict[str, Any]:
        """Generate a tailored resume for a single resume-job pair."""
        start_time = datetime.now()
        prompt = self._build_prompt(resume_text, job_description)
        response, thinking_content = self._call_ollama(prompt)
        end_time = datetime.now()
        
        result = {
            "filename": filename,
            "original_resume": resume_text[:500] + "..." if len(resume_text) > 500 else resume_text,
            "job_description": job_description[:500] + "..." if len(job_description) > 500 else job_description,
            "status": "success",
            "tailored_resume": None,
            "raw_response": None,
            "thinking_content": thinking_content,  # Store the model's reasoning
            "timestamp": end_time.isoformat(),
            "processing_time_seconds": (end_time - start_time).total_seconds()
        }
        
        if response:
            parsed_json = self._extract_json(response)
            if parsed_json:
                result["tailored_resume"] = parsed_json
            else:
                result["status"] = "json_parse_error"
                result["raw_response"] = response[:2000] if len(response) > 2000 else response
        else:
            result["status"] = "api_error"
        
        return result
    
    def process_dataframe(
        self,
        df: pd.DataFrame,
        resume_col: str = "resume_text",
        job_col: str = "job_description",
        filename_col: str = "filename",
        start_idx: int = 0,
        end_idx: Optional[int] = None,
        save_every: int = 10
    ) -> None:
        """
        Process a DataFrame and generate tailored resumes.
        
        Args:
            df: DataFrame with resume and job description columns
            resume_col: Name of the resume text column
            job_col: Name of the job description column
            filename_col: Name of the filename column
            start_idx: Starting index for processing
            end_idx: Ending index for processing (None = process all)
            save_every: Save progress after every N records
        """
        if end_idx is None:
            end_idx = len(df)
        
        df_subset = df.iloc[start_idx:end_idx]
        
        successful = 0
        failed = 0
        
        batch_start_time = datetime.now()
        print(f"{'='*50}")
        print(f"Starting processing at: {batch_start_time.isoformat()}")
        print(f"Model: {self.model_name}")
        print(f"Thinking mode: {'Enabled' if self.enable_thinking else 'Disabled'}")
        print(f"Output file: {self.output_path}")
        print(f"Processing records {start_idx} to {end_idx} ({len(df_subset)} total)")
        print(f"{'='*50}\n")
        
        # Open file in append mode
        with open(self.output_path, 'a', encoding='utf-8') as fh:
            for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Generating resumes"):
                resume_text = row[resume_col]
                job_description = row[job_col]
                filename = row[filename_col]
                
                result = self.generate_single(resume_text, job_description, filename)
                result["original_index"] = idx
                result["session_start"] = self.session_start
                result["model_used"] = self.model_name
                
                # Write to file immediately
                fh.write(json.dumps(result, ensure_ascii=False) + "\n")
                
                if result["status"] == "success":
                    successful += 1
                else:
                    failed += 1
                
                # Flush periodically
                if (successful + failed) % save_every == 0:
                    fh.flush()
        
        batch_end_time = datetime.now()
        total_time = (batch_end_time - batch_start_time).total_seconds()
        
        print(f"\n{'='*50}")
        print(f"Processing complete!")
        print(f"Model:      {self.model_name}")
        print(f"Started:    {batch_start_time.isoformat()}")
        print(f"Finished:   {batch_end_time.isoformat()}")
        print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Successful: {successful}")
        print(f"Failed:     {failed}")
        if successful + failed > 0:
            print(f"Avg time/record: {total_time/(successful+failed):.2f} seconds")
        print(f"Results saved to: {self.output_path}")
        print(f"{'='*50}")
    
    def check_ollama_status(self) -> bool:
        """Check if Ollama is running and the model is available."""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=5)
            response.raise_for_status()
            models = response.json().get("models", [])
            model_names = [m.get("name", "") for m in models]
            model_base_names = [m.split(":")[0] for m in model_names]
            
            print(f"Ollama is running. Available models: {model_names}")
            
            model_base = self.model_name.split(":")[0]
            if self.model_name in model_names or model_base in model_base_names:
                print(f"✓ Model '{self.model_name}' is available")
                if "qwen3" in self.model_name.lower():
                    print(f"  Note: Qwen3 thinking model detected - extended timeout set to {self.timeout}s")
                return True
            else:
                print(f"✗ Model '{self.model_name}' not found. Please run: ollama pull {self.model_name}")
                return False
        except requests.exceptions.RequestException as e:
            print(f"✗ Cannot connect to Ollama at {self.base_url}")
            print(f"  Error: {e}")
            print(f"  Make sure Ollama is running: ollama serve")
            return False


print("OllamaResumeGenerator class defined successfully!")

OllamaResumeGenerator class defined successfully!


In [ ]:
# Initialize the generator and check Ollama status
generator = OllamaResumeGenerator(
    model_name="qwen3:8b",  # Change to your preferred model
    output_path="dataset/tailored_resumes/tailored_resumes",
    timeout=180,
    max_retries=3
)

# Check if Ollama is running
generator.check_ollama_status()

Ollama is running. Available models: ['qwen3-vl:8b', 'qwen3:8b', 'phi4-reasoning:plus', 'gpt-oss:20b', 'deepseek-r1:14b', 'mixtral:8x7b', 'mistral-nemo:latest', 'mistral:7b']
✓ Model 'qwen3:8b' is available
  Note: Qwen3 thinking model detected - extended timeout set to 180s


True

In [30]:
# Process the dataset (start with a small batch to test)
# Uncomment the line below to process all records, or adjust start_idx and end_idx

# Test with first 5 records
generator.process_dataframe(
    df,
    resume_col="resume_text",
    job_col="job_description",
    filename_col="filename",
    start_idx=0,
    end_idx=1,  # Change to None to process all 1565 records
    save_every=1
)

Starting processing at: 2025-12-01T02:23:04.655779
Model: qwen3:8b
Thinking mode: Enabled
Output file: dataset\tailored_resumes_20251201_022302.jsonl
Processing records 0 to 1 (1 total)



Generating resumes: 100%|██████████| 1/1 [00:26<00:00, 26.10s/it]


Processing complete!
Model:      qwen3:8b
Started:    2025-12-01T02:23:04.655779
Finished:   2025-12-01T02:23:30.761733
Total time: 26.11 seconds (0.44 minutes)
Successful: 1
Failed:     0
Avg time/record: 26.11 seconds
Results saved to: dataset\tailored_resumes_20251201_022302.jsonl


In [35]:
# Read and display the generated results
results_df = pd.read_json('dataset\\tailored_resumes_20251201_022302.jsonl', lines=True)
print(f"Generated {len(results_df)} tailored resumes")
print(f"\nStatus distribution:")
print(results_df['status'].value_counts())

# Show a sample of a successful result
successful = results_df[results_df['status'] == 'success']
if len(successful) > 0:
    print(f"\n=== Sample Tailored Resume ===")
    sample = successful.iloc[0]
    print(f"Filename: {sample['filename']}")
    print(f"\nTailored Resume JSON:")
    print(json.dumps(sample['tailored_resume'], indent=2))

Generated 1 tailored resumes

Status distribution:
status
success    1
Name: count, dtype: int64

=== Sample Tailored Resume ===
Filename: 00_Willie_Ellis_Go_Python.docx

Tailored Resume JSON:
{
  "personal_information": {
    "name": "Willie Ellis",
    "email": "willieellis0177@gmail.com",
    "phone": "760 995 2578",
    "location": "Buffalo, New York",
    "socials": [
      {
        "name": "LinkedIn",
        "link": "http://www.linkedin.com/in/willieellisa6b492212"
      }
    ]
  },
  "summary": "Detail-oriented and dependable individual with a strong work ethic, capable of maintaining a safe and organized work environment. Eager to contribute to production goals and ensure quality standards.",
  "experiences": [
    {
      "designation": "Textiles Assistant",
      "companyName": "Goodwill Industries of Northwest NC",
      "location": "Brevard, NC",
      "start_date": "2023-08-01",
      "end_date": "Present",
      "points": [
        "Sorting clothing with attention to q

In [36]:
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   filename                 1 non-null      object        
 1   original_resume          1 non-null      object        
 2   job_description          1 non-null      object        
 3   status                   1 non-null      object        
 4   tailored_resume          1 non-null      object        
 5   raw_response             0 non-null      float64       
 6   thinking_content         0 non-null      float64       
 7   timestamp                1 non-null      datetime64[ns]
 8   processing_time_seconds  1 non-null      float64       
 9   original_index           1 non-null      int64         
 10  session_start            1 non-null      object        
 11  model_used               1 non-null      object        
dtypes: datetime64[ns](1), float64(3), int64(

# Gemini Batch API Code Generation

In [3]:
# JSON Schema for tailored resume output
RESUME_SCHEMA = '''{"$schema":"http://json-schema.org/draft-04/schema#","type":"object","properties":{"personal_information":{"type":"object","properties":{"name":{"type":"string"},"email":{"type":"string"},"phone":{"type":"string"},"location":{"type":"string"},"socials":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]}},"required":["name","email","phone","location"]},"summary":{"type":"string"},"experiences":{"type":"array","items":[{"type":"object","properties":{"designation":{"type":"string"},"companyName":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"points":{"type":"array","items":[{"type":"string"}]}},"required":["designation","companyName","location","start_date"]}]},"education":{"type":"array","items":[{"type":"object","properties":{"institution":{"type":"string"},"degree":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"gpa":{"type":"string"}},"required":["institution","degree","location","start_date","gpa"]}]},"skills":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"data":{"type":"array","items":[{"type":"string"}]}},"required":["name","data"]}]},"projects":{"type":"array","items":[{"type":"object","properties":{"projectName":{"type":"string"},"caption":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"url":{"type":"string"},"projectDetails":{"type":"array","items":[{"type":"string"}]},"externalSources":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]},"technologiesUsed":{"type":"array","items":[{"type":"string"}]}},"required":["projectName","location","projectDetails"]}]},"certifications":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"issuing_organization":{"type":"string"},"issue_date":{"type":"string"},"expiration_date":{"type":"string"},"credential_id":{"type":"string"},"url":{"type":"string"}},"required":["name","issuing_organization","issue_date","expiration_date","credential_id","url"]}]},"awards":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"extracurricular_achievements":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"languages":{"type":"array","items":[{"type":"object","properties":{"language":{"type":"string"},"proficiency":{"type":"string"}},"required":["language","proficiency"]}]}},"required":["personal_information","education","skills","extracurricular_achievements"]}'''


SYSTEM_PROMPT = """You create a tailored resume based on the job description.

Your task:
1. Read the RESUME_TEXT.
2. Read the JOB_DESCRIPTION.
3. Use only the information inside these two.
4. Follow the SCHEMA exactly.
5. Write a tailored resume in JSON using the SCHEMA.
6. Do not output anything outside the JSON.
7. If a field is missing in the resume, write a short, safe placeholder that fits the job.

"""

PROMPT_TEMPLATE = """
RESUME_TEXT:
\"\"\"
<<PASTE RESUME HERE>>
\"\"\"

JOB_DESCRIPTION:
\"\"\"
<<PASTE JD HERE>>
\"\"\"

SCHEMA:
\"\"\"
<<PASTE JSON SCHEMA HERE>>
\"\"\"

Create a tailored resume in JSON following the SCHEMA exactly.
Use only content from the RESUME_TEXT but rewrite it to match the JOB_DESCRIPTION.
Do not add extra lines or explanation.
Output only JSON.
"""

In [4]:
def _build_USER_prompt(resume_text: str, job_description: str) -> str:
    """Build the prompt by filling in the template (Your Custom Logic)."""
    prompt = PROMPT_TEMPLATE.replace("<<PASTE RESUME HERE>>", resume_text)
    prompt = prompt.replace("<<PASTE JD HERE>>", job_description)
    prompt = prompt.replace("<<PASTE JSON SCHEMA HERE>>", RESUME_SCHEMA)
    return prompt

In [ ]:
DATA_DIR = ".//dataset"
df = pd.read_parquet(os.path.join(DATA_DIR, 'resume_dataset.parquet'), engine='pyarrow')
df_subset = df.head(10)  # For testing, use first 10 rows

# Column names from your dataframe
resume_col = "resume_text"
job_col = "job_description"




def generate_request_file(batch_name , df_subset):
    """Generates a JSONL file for a specific model."""
    output_filename = f"batch_requests_{batch_name}.jsonl"
    requests = []
    print(f"\nGenerating requests for  -> {output_filename}")
    # Your loop logic using tqdm
    count = 0
    for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=f"Processing {batch_name}"):
        count += 1
        resume_text = row[resume_col]
        job_description = row[job_col]

        # 1. Create the full user prompt
        user_content = _build_USER_prompt(resume_text, job_description)

        # 2. Build the Batch API Request Object
        # Note: The 'url' must point to the specific model version
        request_entry = {
            "key": f"request-{batch_name}-idx{idx}-{count}", # Unique ID for this request
            
            "request": {
                "contents": [
                    {"role": "user", "parts": [{"text": user_content}]}
                ],
                "system_instruction": {
                    "parts": [{"text": SYSTEM_PROMPT}]
                },
                "generationConfig": {
                    "responseMimeType": "application/json",
                    "temperature": 0.2
                }
            }
        }
        requests.append(request_entry)
    
    output_path = Path(f"dataset/batch_requests/{output_filename}")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    # 3. Write to JSONL file
    with output_path.open('w', encoding='utf-8') as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")
            
    print(f"Success: Created {output_filename} with {len(requests)} requests.")

# Run generation for all 3 models
key = "batch-4"
df_subset = df.iloc[521:1565]  # Adjust indices as needed
generate_request_file(key, df_subset)


Generating requests for  -> batch_requests_batch-4.jsonl


Processing batch-4: 100%|██████████| 1044/1044 [00:00<00:00, 11106.29it/s]

Success: Created batch_requests_batch-4.jsonl with 1044 requests.


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1565 entries, 0 to 2126
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   filename         1565 non-null   object
 1   filetype         1565 non-null   object
 2   resume_text      1565 non-null   object
 3   has_images       1565 non-null   bool  
 4   job_description  1565 non-null   object
dtypes: bool(1), object(4)
memory usage: 62.7+ KB


In [ ]:
# Configure GenAI client with Gemini API key
client = genai.Client(
        api_key="",
    )
print("GenAI configured successfully with gemini api key!")

GenAI configured successfully with gemini api key!


In [ ]:
import os

# Set the path to your Google Cloud service account key file
# Replace "path/to/your/service-account-key.json" with the actual path to your file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "C:\\Users\\Documents\\Repo\\conda\\harsha-proj\\gen-lang-client-0465995346-fd97340f8167.json"


In [115]:
# Configure GenAI client for Vertex AI
client = genai.Client(
        vertexai=True, project='gen-lang-client-0465995346', location='us-central1'
    )
print("GenAI configured successfully with vertex api key!")

GenAI configured successfully with vertex api key!


In [108]:
batch_file = 'dataset/batch_requests/batch_requests_batch-4.jsonl'

In [ ]:
uploaded_file = client.files.upload(
    file= batch_file,
    config=types.UploadFileConfig(display_name='my-batch-requests', mime_type='jsonl')
)

print(f"Uploaded file: {uploaded_file.name}")


# test file : Uploaded file: files/uno9vg4u5mtt   Created batch job: batches/k0wqlp3jwrp85s8xf06y1v0hp222qb27hd5t
# bacth-1 : Uploaded file: files/igujaoz0own5     Created batch job: batches/mpfcsvvz5f3demju35es8w4tlqzunsjncxhn
# bacth-2 : Uploaded file: files/r5wimsjr6vds or Uploaded file: files/nn7df9hpccae
# bacth-3 : 
# bacth-4 : Uploaded file: files/saw7wfyrzg3p

In [116]:
display_name = f'batch-upload-job-batch_requests_batch-4'
file_name = 'gs://harsha-dump/batch_requests_batch-4.jsonl' 

In [118]:

file_batch_job = client.batches.create(
    model="gemini-2.5-flash",
    src=file_name,
    config={
        'display_name': display_name,
    },
)

print(f"Created batch job: {file_batch_job.name}")

Created batch job: projects/982088254342/locations/us-central1/batchPredictionJobs/8949234702531690496


In [131]:
# job_name = "batches/k0wqlp3jwrp85s8xf06y1v0hp222qb27hd5t"  # (e.g. 'batches/your-batch-id')
job_name = "projects/982088254342/locations/us-central1/batchPredictionJobs/2288129378674016256"
batch_job = client.batches.get(name=job_name)

completed_states = set([
    'JOB_STATE_SUCCEEDED',
    'JOB_STATE_FAILED',
    'JOB_STATE_CANCELLED',
    'JOB_STATE_EXPIRED',
])

print(f"Polling status for job: {job_name}")
batch_job = client.batches.get(name=job_name) # Initial get
count = 0

print(f"Current state: {batch_job.state.name}")

if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"Error: {batch_job.error}")

Polling status for job: projects/982088254342/locations/us-central1/batchPredictionJobs/2288129378674016256
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING


In [127]:
client.batches.cancel(name=job_name)

In [ ]:
job_name = "batches/mpfcsvvz5f3demju35es8w4tlqzunsjncxhn"  # (e.g. 'batches/your-batch-id')

batch_job = client.batches.get(name=job_name)

if batch_job.state.name == 'JOB_STATE_SUCCEEDED':

    # If batch job was created with a file
    if batch_job.dest and batch_job.dest.file_name:
        # Results are in a file
        result_file_name = batch_job.dest.file_name
        print(f"Results are in file: {result_file_name}")

        print("Downloading result file content...")
        file_content = client.files.download(file=result_file_name)
        # Process file_content (bytes) as needed
        print(file_content.decode('utf-8'))

        output_path = Path(f"dataset/batch_results/{result_file_name.replace('/', '_')}.jsonl")
        output_path.parent.mkdir(parents=True, exist_ok=True)
         # Adjust path as needed
        
        with output_path.open('wb') as f:
            f.write(file_content)
        print(f"File content saved to {output_path}")

    # If batch job was created with inline request
    # (for embeddings, use batch_job.dest.inlined_embed_content_responses)
    elif batch_job.dest and batch_job.dest.inlined_responses:
        # Results are inline
        print("Results are inline:")
        for i, inline_response in enumerate(batch_job.dest.inlined_responses):
            print(f"Response {i+1}:")
            if inline_response.response:
                # Accessing response, structure may vary.
                try:
                    print(inline_response.response.text)
                except AttributeError:
                    print(inline_response.response) # Fallback
            elif inline_response.error:
                print(f"Error: {inline_response.error}")
    else:
        print("No results found (neither file nor inline).")
else:
    print(f"Job did not succeed. Final state: {batch_job.state.name}")
    if batch_job.error:
        print(f"Error: {batch_job.error}")

## Cleaning the batch process code

In [176]:
df['id']=df.index

In [ ]:
DATA_DIR = ".//dataset"

df.to_parquet(os.path.join(DATA_DIR, 'resume_dataset.parquet'), engine='pyarrow', compression='snappy')

df = pd.read_parquet(os.path.join(DATA_DIR, 'resume_dataset.parquet'), engine='pyarrow')

In [211]:
batch_result_path_1 = f'.//dataset//batch_results//files_batch-mpfcsvvz5f3demju35es8w4tlqzunsjncxhn.jsonl'
batch_result_path = f'.//dataset//batch_results//batch-output_prediction-model-2025-12-02T21_47_49.533170Z_predictions.jsonl'
list_of_dfs = []
temp = pd.read_json(batch_result_path_1, lines=True)
list_of_dfs.append(temp)
print(f"Loaded {len(temp)} records from first batch result")
temp_2 = pd.read_json(batch_result_path, lines=True)
list_of_dfs.append(temp_2)
print(f"Loaded {len(temp_2)} records from second batch result")
results_df = pd.concat(list_of_dfs, ignore_index=True)

results_df = results_df.drop(columns=['request', 'status','processed_time'])

results_df.info()
results_df.head()

Loaded 521 records from first batch result
Loaded 1044 records from second batch result
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1565 entries, 0 to 1564
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   response  1565 non-null   object
 1   key       1565 non-null   object
dtypes: object(2)
memory usage: 24.6+ KB


,response,key
0,"{'candidates': [{'index': 0, 'finishReason': '...",request-batch-1-idx0-1
1,"{'responseId': 'FEUvaf3SLsnYqtsPka-dGA', 'usag...",request-batch-1-idx1-2
2,"{'candidates': [{'index': 0, 'finishReason': '...",request-batch-1-idx2-3
3,"{'responseId': 'FUUvabHPMOD6qtsPicKv-QY', 'mod...",request-batch-1-idx3-4
4,"{'candidates': [{'finishReason': 'STOP', 'cont...",request-batch-1-idx6-5


In [212]:
def extract_indices(string: str) -> Optional[int]:
    """
    Extracts the digits immediately following the 'idx' substring using a
    capture group.
    """
    # Pattern looks for 'idx' and CAPTURES the digits (\d+) that follow
    pattern = re.compile(r'idx(\d+)') 
    results = None
    
    match = pattern.search(string)
    if match:
        # *** FIX IS HERE ***
        # Use match.group(1) to get ONLY the captured digits ('1944' or '716')
        index_value = int(match.group(1)) 
        results = index_value

    return results

In [214]:
def _extract_json(response: str) -> Optional[Dict[str, Any]]:
    """Extract JSON from the model response."""
    if not response:
        return None
    
    # Try to find JSON in the response
    response = response.strip()
    
    # Try direct parsing first
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        pass
    
    # Try to extract JSON from markdown code blocks
    if "```json" in response:
        start = response.find("```json") + 7
        end = response.find("```", start)
        if end > start:
            try:
                return json.loads(response[start:end].strip())
            except json.JSONDecodeError:
                pass
    
    # Try generic code blocks
    if "```" in response:
        start = response.find("```") + 3
        # Skip language identifier if present
        newline_pos = response.find("\n", start)
        if newline_pos > start:
            start = newline_pos + 1
        end = response.find("```", start)
        if end > start:
            try:
                return json.loads(response[start:end].strip())
            except json.JSONDecodeError:
                pass
    
    # Try to extract JSON between curly braces
    start = response.find("{")
    end = response.rfind("}") + 1
    if start >= 0 and end > start:
        try:
            return json.loads(response[start:end])
        except json.JSONDecodeError:
            pass
    
    return None


In [216]:
# Process batch results and map back to original dataframe
processed_results = []
errors = []

for idx, row in tqdm(results_df.iterrows(), total=len(results_df), desc="Mapping original indices"):
    key = row['key']
    extracted_index = extract_indices(key)
    
    result = {
        'key': key,
        'original_index': extracted_index,
        'status': 'success',
        'tailored_resume': None,
        'raw_response': None,
        'error': None
    }
    
    try:
        # Safely navigate the response structure
        response = row.get('response', {})
        candidates = response.get('candidates', [])
        
        if not candidates:
            result['status'] = 'no_candidates'
            result['error'] = 'No candidates in response'
        else:
            content = candidates[0].get('content', {})
            parts = content.get('parts', [])
            
            if not parts:
                result['status'] = 'no_parts'
                result['error'] = 'No parts in content'
            else:
                text = parts[0].get('text', '')
                result['raw_response'] = text
                
                # Try to extract JSON from the response
                cleaned_json = _extract_json(text)
                if cleaned_json:
                    result['tailored_resume'] = cleaned_json
                    result['status'] = 'success'
                else:
                    result['status'] = 'json_parse_error'
                    result['error'] = 'Failed to parse JSON from response'
                    
    except Exception as e:
        result['status'] = 'error'
        result['error'] = str(e)
        errors.append({'key': key, 'error': str(e)})
    
    # Check if index exists in original dataframe
    if extracted_index is not None and extracted_index in df.index.tolist():
        result['found_in_df'] = True
    else:
        result['found_in_df'] = False
    
    processed_results.append(result)

# Create a dataframe from processed results
processed_df = pd.DataFrame(processed_results)

# Display summary
print(f"\n=== Processing Summary ===")
print(f"Total records: {len(processed_df)}")
print(f"Status distribution:")
print(processed_df['status'].value_counts())
print(f"\nFound in original df: {processed_df['found_in_df'].sum()}")
print(f"Not found in original df: {(~processed_df['found_in_df']).sum()}")

if errors:
    print(f"\n=== Errors ({len(errors)}) ===")
    for err in errors[:5]:  # Show first 5 errors
        print(f"Key: {err['key']}, Error: {err['error']}")

Mapping original indices: 100%|██████████| 1565/1565 [00:00<00:00, 8792.20it/s]


=== Processing Summary ===
Total records: 1565
Status distribution:
status
success             1530
json_parse_error      22
no_parts              13
Name: count, dtype: int64

Found in original df: 1565
Not found in original df: 0


In [ ]:
# Display sample of successful results
successful = processed_df[processed_df['status'] == 'success']
print(f"Successful extractions: {len(successful)}")

Successful extractions: 1530

=== Sample Tailored Resume ===
Key: request-batch-1-idx0-1
Original Index: 0

Tailored Resume JSON (first 500 chars):
{
  "personal_information": {
    "name": "Willie Ellis",
    "email": "willieellis0177gmailcom",
    "phone": "760 995 2578",
    "location": "Buffalo New York",
    "socials": [
      {
        "name": "LinkedIn",
        "link": "httpswwwlinkedincominwillieellisa6b492212"
      }
    ]
  },
  "summary": "Highly motivated professional with 7 years of work experience, demonstrating strong commitment, dependability, and hard work. Proven ability to implement process improvements to optimize effi...


In [ ]:
# Add tailored_resume column to the original dataframe
# Initialize the column with None
df['tailored_resume'] = None

count = 0
not_found = 0

for idx, row in tqdm(successful.iterrows(), total=len(successful), desc="Mapping tailored resumes to df"):
    original_idx = row['original_index']
    
    # Check if the original_index exists in df's id column
    if original_idx in df['id'].values:
        # Set the tailored_resume for matching rows
        df.loc[df['id'] == original_idx, 'tailored_resume'] = [row['tailored_resume']]
        count += 1
    else:
        not_found += 1

print(f"\n=== Mapping Summary ===")
print(f"Successfully mapped: {count}")
print(f"Not found in df: {not_found}")
print(f"Total rows with tailored_resume: {df['tailored_resume'].notna().sum()}")

print(f"Before removing nulls: {len(df)} rows")
df = df[df['tailored_resume'].notna()]
print(f"After removing nulls: {len(df)} rows")
df.drop(columns=['has_images'], inplace=True)
df.info()


Mapping tailored resumes to df: 100%|██████████| 1530/1530 [00:00<00:00, 2359.29it/s]


=== Mapping Summary ===
Successfully mapped: 1530
Not found in df: 0
Total rows with tailored_resume: 1530
Before removing nulls: 1565 rows
After removing nulls: 1530 rows


## Saving the final dataset

In [233]:
DATA_DIR = ".//dataset"

df.to_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow', compression='snappy')
df.to_json(os.path.join(DATA_DIR, 'final_resume_dataset.jsonl'), orient='records', lines=True, force_ascii=False)
df.to_csv(os.path.join(DATA_DIR, 'final_resume_dataset.csv'), index=False, encoding='utf-8-sig')

df = pd.read_parquet(os.path.join(DATA_DIR, 'resume_dataset.parquet'), engine='pyarrow')

# Making train.jsonl for finetuning


In [2]:
DATA_DIR = ".//dataset"
df = pd.read_parquet(os.path.join(DATA_DIR, 'resume_dataset.parquet'), engine='pyarrow')

# Base Model Loading and testing

In [3]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import importlib.util

model_name = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Check if flash_attn is available
flash_attn_available = importlib.util.find_spec("flash_attn") is not None
use_flash_attn = flash_attn_available and torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2" if use_flash_attn else "eager",
    torch_dtype=torch.bfloat16,
)

# prepare the model input
messages = [
    {"role": "system", "content": "You create a tailored resume based on the job description.\n\nYour task:\n1. Read the RESUME_TEXT.\n2. Read the JOB_DESCRIPTION.\n3. Use only the information inside these two.\n4. Follow the SCHEMA exactly.\n5. Write a tailored resume in JSON using the SCHEMA.\n6. Do not output anything outside the JSON.\n7. If a field is missing in the resume, write a short, safe placeholder that fits the job.\n\n"},
    {"role": "user", "content": "\nRESUME_TEXT:\n\"\"\"\nPHANI SETTY 214 9235723 settyphanigmailcom Summary A handson programmer in userinterface design and development with over 20 years of experience and a proven track record in shipping web experiences for singlepage applications hybrid apps online stores and interacting market content Extensive experience in Architecting and developing for large and distributed frontend codebases as well as restructuring legacy systems to reduce bloat increase modularity and speed up future development iterations I have worked with a variety of web application stacks along with their respective view and templating systems When I am not coding I am involved in UXside of things like sketching out wireframes and designs on paper or whiteboard creating highfidelity designs and clickable prototypes and usertesting them I have developed innovative products and express brands through strategically driven design and interactive models 15 years of handson experience in leading teams building highperforming teams of onsite remote and offshore developers and designers Afterhours Im an avid learner of new technologies through personal projects as well as writing a book I was a Certified PMP20062009 and a Certified ScrumMasterpresent I bring people and technology together through effective communication and leadership Im equally comfortable discussing business requirements with product managers architects and APIs with developers Ive designed developed and delivered technology solutions in both traditional Waterfall and Agile development environments as well as transitioning between them I Have worked extensively in Healthcare Education Management Consulting domains I have strong technical and communication skills Core Competencies CSS Architecture I Frontend Architecture I Design Systems I Adaptive responsive design I wireframing Prototyping browser device debugging I Pageload Performance tuning I Mentoring and Training I Agile software development Technical Competencies Clientside UI JavaScript JQuery ReactJS redux Angular Websockets RequireJS HeatmapJS TypeScript HTMS CSS pre post processing SVG Backbone Handlebars D3 Environments ASPNET Core Web Forms and MVC Nodejs Java Grails PHP Git Github Gitlab CVS TFS Tools Visual Studio Code IntelliJ Adobe CC suite Education BE Computer Science Amravati University 1996 BSIT from Grantham University2017 MSSoftware Engineering from Walden University2021 Certifications Certified scrum master2018 Certified PMP2006 Six sigmagreen belt 2002 Projects CBRE UI Lead July 2018 Till Date Dallas tx Responsibilities As the onshore lead for a large consulting team I oversaw coding standards related to Angular 28 CSS JavaScript I wrote prototypes as well as augmented teams to facilitate project completion My UI organization included creating and administering training programs and overseeing code reviews for a global team of developers at various skill levels along with working closely with clients being empathetic to their needs all the while interpreting them in a cognitive experience that makes sense to their customer Accomplished Web project objectives by establishing clear understanding of project requirements Involved in designing and developing the components using HTML CSS JavaScript Bootstrap SASS Angular8 Flex and NodeJS Involved in implementing various screens for the frontend using Angular and used various public libraries from NPM Node Package Manager Collaborate crossfunctionally to develop research plan user flows information architecture and wireframes and work with Agile Scrum teams Managed a growing team of UI Engineers focused on building great experiences teaching and implementing excellence Visualize large data and develop dashboards As a lead Have good ability to analyze problems find solutions and implement them to tight deadlines on time I have good experience writing technical briefs technical specifications and generating costingtimings for projects Dealer socket Tech Leadui apr2018 July 2018 irving tx Responsibilities As a Lead I am responsible for making sure a quality and a welldocumented and test driven code is developed Extensive experience in building many software solutions with particular emphasis on clientside code Web Apps and Native Mobile Apps Specialized in architecting UI frameworks and creating custom reusable user interface components Create responsive landing pages and email templates for product communications Create and manage paid social media ad campaigns to drive lead generation Develop maintain systems utilizing HTML CSS JavaScriptjQuery Develop maintain systems utilizing client side frameworks and libraries Work closely with product owners backend C developers and UX designers to implement new features Mobile Applications Web Single Page Applications UI Components development WebComponents React Redux ReduxSaga ES Flow Jest Enzyme Babel NodeJS Webpack CI with Bitbuckets Pipeline and Docker containers HarMAN internationalTech leadui Dec2017 April 2018Plano tx Responsibilities As a Lead I am responsible for making sure a quality and a welldocumented code is developed Prototypingarchitecting and implementation of UI Secure clustering components and pages using ReactRedux Bootstrap UI RESTfetch Knex Bookshelf SASS GitGithub Webpack CreatingRefactoring ReactRedux reusable components for integration into encrypting dependencies Design and develop reusable Angular 4 Node Package Modules Provide guidance on technical architecture using HTML5JavaScript web technologies Builds responsive web user interfaces that ensure seamless user experience across desktop and mobile platforms Builds fullduplex embedded web applications that control hardware devices in realtime using web sockets Establishes UI development process for embedded devices Unit testing with all elements of ReactRedux project by using Jest Enzyme Adjustment desktop web apps for mobilesize devices such as smartphones and tablets Mock servers coding on Python and Golang for unit testing on local environment Integration testing with Selenium IBMUI ArchitectJuly 2017 dec2017 Santa Clara CA Responsibilities DesignArchitect support and lead both offshore and onsite teamsabout 10 React and AngularUI developers as part of refactoring an IBMs flagship product Worked with modules like MongoDB and mongoose for database persistence using Nodejs to interact with mongodb Worked with unit testing of javascript applications using Karma Jasmine apimocker Jest enzyme snion Tasks include making sure a quality and a welldocumented code is developed and ESLint errors are fixed unit tests are written and the CI builds are through for code merge Interpret clients needs and ability to architect design and develop solutions with high visual impact to get the clients message across I am responsible for user interface strategy planning development and delivery across the practice My duties are to maintain consistent design guidelines best practices and standards and project methodologies My role as the Architect for UI is to work with product owners and stakeholders to present and create solutions to visualize design and deliver style guides prototypes and assets for the client with full compliance of client rules and guidelines Optum technologiesuhg dev LEAD April2016 May 2017 Minneapolis Plano Responsibilities Evaluate and implement SEO friendly infrastructures in HTML and Javascript to new and existing applications Migrating legacy Angular 14 components to higher versions at the moment Front end architecture and development of largescale Angular application in the AEM environment Developing a scaffolding system to build the frontend using Gulp along with integrating SASS preprocessing and Bootstrap library SPAs using AngularJS 1x including factory services and directives consumption of web services Consulting on UX and visual design options Recommending design and technical solutions based on user requirements and business needs Develop poling and cross team communication ApplicationPCTC Developed documentation testing standards Implementation of Bootstrap Foundation and Jquery frameworks HealthPartners Senior Web DeveloperUI Lead Jan 2016 March 2016 Minneapolis HealthPartners is an integrated nonprofit health care provider and health insurance company located in Bloomington Minnesota offering care coverage research and education to its members patients and the community Duties include working collaboratively with the team and management on designing and delivering complex crossbrowser applications and maintaining existing products Responsibilities I was programming Widgets and Applications for wwwHealthPartnerscom using Angularjs Javascript and BootStrap Design development and testing phases of Software Development using AGILE Methodology and Test Driven Development TDD Involvement in all stages of Software development life cycle including Analysis development Implementation testing and support Involved in development of User Interface using HTML5 CSS3 JavaScript and jQuery AJAX JSON Developed single page web application using JavaScript framework Worked with CrossBrowser Compatible issues Created reusable templates and style sheets based on UI standards and guidelines Performed Functional tasks using specifications and wireframes Extensively used Debugging Cascading Style Sheets to change the styles now and in the future Designed and implemented the UI with extensive use of JavaScript JSON and Ajax Designed and developed basic user interfaces to HealthPartners web services by analyzing business requirements and priorities Provides code and web design reviews Integrated applications to web services via server scripting and database architecting Troubleshoots development and production incidents across multiple environments and operating platforms Medtronic UI Lead Nov 2014 Dec 2015 Minneapolis Responsibilities I was responsible for Architecting used combination of Event MVC and AMD patterns designing and programming the frontend for an Electro Cradio Gram waveforms rendering mobile application using Javascript and HTML 5 canvas I have developed a handy UI widget library using pure Javascript for our internal app developers to create UI elements dynamically Managed software development operational and client integration projects Created a complete test bed for the UI usingstubbing the server using Node JS I have designed the widget library in the AMD pattern using RequireJS I have also developed various reports and charts using HTML Canvas HTML SVG D3JS and SVGjs by passing JSON objects or Arrays as input both for mobile and web applications I have worked extensively on Ajax and JavaScript Websockets I have revamped an existing single thread application that had heavy computational data in the UI to a light weight application using Web workers Have been working on Jasmine and Chutzpah for my unit testing as we develop using TDD approach in Visual StudioXamarin environment Worked extensively with jQuery HTML4 and CSS Developed prototypes and mokups using Adobe Fireworks Edge and Balsamiq Developed user friendly and attractive UI Have also hand coded web templates using HTML5 Javascript Bootstrap and CSS3 Harvard Business Publishing Global engagement manager UI Lead Sep 2010 October 2014 Boston MA Responsibilities Managing the global vendors offering services in content development and translationlocalization Gathering and analyzing requirements and writing SOWs and scope documents along with leading the project from concept to completion Responsible for User Interface design and development for Harvard Business Publishings portals and products like LeadershipDirectorg WCMS and Harvard Manage Mentor using Adobe Photoshop CS5 Adobe Flash CS5 Illustrator HTML5 Javascript Angular JS JQuery CSS3 and AJAX I was responsible for designing solutions that improved user experience and supported graphic resource needs for various products for Harvard Business Publishing Boston MA I was also responsible for creatinguser personas creating wireframes visual mockups UI specs and flash designs for service window applications Created design strategy and implemented in various UIUX projects I have conducted research and data evaluation on interactive products and created graphics animations using flash and after effects for various service window applications Responsible for reviewing creative work provide art direction and design feedback when working with junior designers and developers My technical and creative Skills helped me to work on multiple projects simultaneously I have worked with product management and engineering teams to ensure that the graphics and layout designs meet customer requirements and implementation constraints GE Sr UI Developerproject managerTech lead Oct 2004 Sep 2010 Albany New York Responsibilities Gathering and analyzing requirements and writing SOWs and scope documents along with leading the project from concept to completion Responsible for User Interface Design for web applications and learning portals using Adobe Photoshop CS5 Adobe Flash CS5 Illustrator HTML5 Javascript CSS3 AJAXand the clients preferred content management tools like Knet Participate in early sprints of agile methods Work ahead of sprint and keep the work ready for the development team for development Created the rich user experience and user interface design for their Archeological data collection application for constructions process Involved in the hiring process and recruited worldclass talent user interface development group Support flash action script for interactive service window applications across the apps team Involved in the research and discover phase of the apps development for client and user research Created the rich presentations for the corporate initiatives United Nations Development Program UI Dev Jul 2002 Sep 2004 New York Responsibilities Responsible for User Interface Design using Adobe Photoshop Flash HTML CSS JavaScript and the companys custom web development and content management tools Involve in daily scrums create IA Visual design for agile process Supported interaction design visual design for web products and produced images banner logo poster brochure using softwares like Photoshop Image ready Illustrator In Design IMI Web designer Jan 1999 Jun 2002 Hyderabad India Responsibilities Responsible for User Interface Design and development for various mobilewebdesktop applications for Vodafone I have also created the interface for WAP based application called VOOP Virtual Office on Phone the application enables the users to remotely access their PCs from mobiles to send mails edit documents etc I have also produced entire UI kit for transmission tower designing applications using Photoshop and icon maker tools This was challenging back then as these tools were coded in VC and VC supported transparency only in 256 color ico formats Fountainhead Design Studios Graphic Designer Jan 1997 Jan 1999 Hyderabad India Responsibilities I have created 120 animated greeting cards for Archies online portal using Macromedia Flash 40 on iMac I was involved in the process right from conceptualizing to completion of the greeting cards I have also produced about 150 printed greeting cards using Photoshop Bryce 3D and Corel Draw I have also designed and developed more than 50 websites using basic HTML Macromedia Flash 40 and Dreamweaver 20\n\"\"\"\n\nJOB_DESCRIPTION:\n\"\"\"\nExtra Help Stagehand - 50 vacancies jobs in United StatesOverviewCompanyThis job has closed.APPLY to similar jobsUniversity of Illinois Springfield · 4 weeks agoExtra Help Stagehand - 50 vacanciesSpringfield, ILFull-timeOnsiteEntry Level$20.07/hr - $20.07/hrThe University of Illinois Springfield is seeking Extra Help Stagehands to support their theatrical productions. The role involves coordinating and executing various tasks related to the preparation, operation, and maintenance of event equipment, stage lighting, sound systems, and theatrical scenery, ensuring a safe and efficient working environment.EducationUniversitiesResponsibilitiesAttendance (paid) at employer-required safety training sessionsLoad & unload theatrical equipment into and out of vehicles as required including stacking and unstacking of equipmentMove theatrical equipment into and out of venue storage areasPerform all work required to operate and maintain the safety, trim, balance, and proper rigging of the counterbalanced fly system and associated winched cables, pin-rail equipment and pick-linesPerform such technical specialties as the splicing of cables and ropes, the use of stage weights and braces, the maintenance and correct use of tie lines and pick lines, the maintenance and repair of curtains, and the periodic inspection of curtains in storage to prevent damage.Operate mechanical systems such as pit-lifts, orchestra shell winches, chain motors, etc. Operating of chain motors does NOT also include functions performed by EXTRA HELP RIGGERS.Operate personnel-lift devicesHang, connect to the dimming system, lamp, focus and color theatrical lighting instruments for general and specific usesMove and place audio gear such as microphones, speakers, monitors, control boards and cable as requiredInstall, remove, operate and reconfigure stage curtains, portable flooring, scenery, props and other theatrical items as required.Sweeping and mopping of stage area as required for safety and proper audience presentationMaintain, move, set up and operate spotlights as requiredMaintain and operate lighting and audio control systemsInstall, maintain and operate as needed equipment hung from counter-weighted battens.Setup, maintain (including laundry), repair and place back into storage costumes, wigs and other items worn by performers including towels and associated fabric items. Also assist performers with putting on and taking off costume items.Perform other duties commonly associated with stage operations as requiredQualificationTheatrical lighting operationSound system operationStage riggingCostume maintenancePhysical staminaSafety complianceTeam collaborationAttention to detailRequiredBasic entry-level knowledge of a back-stage theatrical working environment including terminology & standard operating procedures usually acquired through formal education in the performing arts field or previous volunteer/work experience as a stage crew member in a high school, college, community theatre, music club, regional theatre, or professional performance venue.Normal ability to hear, see, speak, smell in a backstage environment which at times can be darkened, crowded, and noisy with multiple hazards.Ability to move quickly unaided.Ability to climb stairs, ladders and work at heights.Ability to lift and move heavy objects.Ability to follow detailed or general directions on how to perform tasks.Ability to work varying and unusual work schedules.Ability to follow and comply with employer's safety rules and regulations.CompanyUniversity of Illinois SpringfieldUniversity of Illinois Springfield, one of three universities in the world-class U of I system, is known for educating public servants and leaders.Founded in 1969Springfield, Illinois, USA501-1000 employeeshttp://www.uis.edu/FundingCurrent StageLate StageLeadership TeamTulio LlosaCIORecent NewsGovernment Technology USIllinois Universities Expand Online Education With Risepoint2025-06-28SlashGear6 Myths About SUVs You Need To Stop Believing2024-11-24StartuptoEnterpriseBreakthrough in Type 1 Diabetes Got iLet Bionic Pancreas System2022-05-14Company data provided by crunchbaseBoost Your Interview ChancesImprove Resume Match ScoreFREE?Your Score8.6Top ApplicantsMust-Have Skills for This RoleTheatrical lighting operationSound system operationStage riggingCostume maintenancePhysical staminaOptimize my ResumeGet Referral Via linkedIn FREE3× Higher Response via Email OutreachLLinda L.Technical Recruiter  \n\"\"\"\n\nSCHEMA:\n\"\"\"\n{\"$schema\":\"http://json-schema.org/draft-04/schema#\",\"type\":\"object\",\"properties\":{\"personal_information\":{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"email\":{\"type\":\"string\"},\"phone\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"socials\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"link\":{\"type\":\"string\"}},\"required\":[\"name\",\"link\"]}]}},\"required\":[\"name\",\"email\",\"phone\",\"location\"]},\"summary\":{\"type\":\"string\"},\"experiences\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"designation\":{\"type\":\"string\"},\"companyName\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"start_date\":{\"type\":\"string\"},\"end_date\":{\"type\":\"string\"},\"points\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]}},\"required\":[\"designation\",\"companyName\",\"location\",\"start_date\"]}]},\"education\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"institution\":{\"type\":\"string\"},\"degree\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"start_date\":{\"type\":\"string\"},\"end_date\":{\"type\":\"string\"},\"gpa\":{\"type\":\"string\"}},\"required\":[\"institution\",\"degree\",\"location\",\"start_date\",\"gpa\"]}]},\"skills\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"data\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]}},\"required\":[\"name\",\"data\"]}]},\"projects\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"projectName\":{\"type\":\"string\"},\"caption\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"start_date\":{\"type\":\"string\"},\"end_date\":{\"type\":\"string\"},\"url\":{\"type\":\"string\"},\"projectDetails\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]},\"externalSources\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"link\":{\"type\":\"string\"}},\"required\":[\"name\",\"link\"]}]},\"technologiesUsed\":{\"type\":\"array\",\"items\":[{\"type\":\"string\"}]}},\"required\":[\"projectName\",\"location\",\"projectDetails\"]}]},\"certifications\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"issuing_organization\":{\"type\":\"string\"},\"issue_date\":{\"type\":\"string\"},\"expiration_date\":{\"type\":\"string\"},\"credential_id\":{\"type\":\"string\"},\"url\":{\"type\":\"string\"}},\"required\":[\"name\",\"issuing_organization\",\"issue_date\",\"expiration_date\",\"credential_id\",\"url\"]}]},\"awards\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"type\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"date\":{\"type\":\"string\"},\"description\":{\"type\":\"string\"}},\"required\":[\"name\",\"type\",\"location\",\"date\",\"description\"]}]},\"extracurricular_achievements\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"name\":{\"type\":\"string\"},\"type\":{\"type\":\"string\"},\"location\":{\"type\":\"string\"},\"date\":{\"type\":\"string\"},\"description\":{\"type\":\"string\"}},\"required\":[\"name\",\"type\",\"location\",\"date\",\"description\"]}]},\"languages\":{\"type\":\"array\",\"items\":[{\"type\":\"object\",\"properties\":{\"language\":{\"type\":\"string\"},\"proficiency\":{\"type\":\"string\"}},\"required\":[\"language\",\"proficiency\"]}]}},\"required\":[\"personal_information\",\"education\",\"skills\",\"extracurricular_achievements\"]}\n\"\"\"\n\nCreate a tailored resume in JSON following the SCHEMA exactly.\nUse only content from the RESUME_TEXT but rewrite it to match the JOB_DESCRIPTION.\nDo not add extra lines or explanation.\nOutput only JSON.\n"},
]

# Apply chat template and tokenize properly
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

# Tokenize the text
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=4096,
    do_sample=False
)

# Get only the generated tokens (exclude input)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# Decode the output
content = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

print("Generated Resume:")
print(content)

c:\ProgramData\anaconda3\envs\finetune\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.06s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated Resume:
{
  "personal_information": {
    "name": "Phani Setty",
    "email": "settyphanigmailcom",
    "phone": "214 9235723",
    "location": "Springfield, IL",
    "socials": []
  },
  "summary": "Entry-level stagehand with hands-on experience in theatrical operations, equipment handling, and stage maintenance. Proven ability to operate stage lighting, sound systems, rigging equipment, and manage stage setups with strong attention to safety, detail, and team collaboration. Experienced in equipment loading/unloading, rigging, cable splicing, and maintaining stage environments. Committed to safety compliance and efficient stage operations in high-pressure environments.",
  "experiences": [
    {
      "designation": "Extra Help Stagehand",
      "companyName": "University of Illinois Springfield",
      "location": "Springfield, IL",
      "start_date": "2025-01-01",
      "end_date": "Present",
      "points": [
        "Load and unload theatrical equipment into and out of 

In [5]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

with open(f'generated_resume_{timestamp}.txt', 'w', encoding='utf-8') as f:
    f.write(content)

print("\nContent saved to 'generated_resume.txt'")


Content saved to 'generated_resume.txt'
